<a href="https://colab.research.google.com/github/shantaislamroza/FlyrankML/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shantaislamroza/FlyrankML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
!pip install duckdb huggingface_hub -q

import os
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
os.environ["HF_TOKEN"] = hf_token

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")


con.execute(f"""
CREATE OR REPLACE SECRET (
    TYPE HUGGINGFACE,
    TOKEN '{hf_token}'
);
""")

print("DuckDB & Hugging Face setup complete!")

DuckDB & Hugging Face setup complete!


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.execute("""
SELECT *
FROM 'hf://datasets/FlyRank/internship-warehouse/*/*2026-03*/*.parquet'
LIMIT 1;
""").df().columns.tolist()

['report_date',
 'client_hash_id',
 'content_hash_id',
 'client_has_gsc',
 'client_has_ga4',
 'gsc_data_available',
 'ga4_data_available',
 'gsc_impressions',
 'gsc_clicks',
 'gsc_sum_position',
 'gsc_avg_position',
 'ga4_pageviews',
 'ga4_sessions',
 'ga4_users',
 'ga4_engaged_sessions',
 'ga4_total_engagement_sec',
 'sessions_organic',
 'sessions_direct',
 'sessions_referral',
 'sessions_social',
 'sessions_paid',
 'sessions_ai',
 'ai_chatgpt',
 'ai_perplexity',
 'ai_gemini',
 'ai_copilot',
 'ai_claude',
 'ai_meta',
 'ai_other',
 'scroll_events',
 'month']

- **Unit of Analysis (Grain):** One row represents a single piece of content (`content_hash_id`) for a specific client (`client_hash_id`) on a specific report date (`report_date`).
- **Table:** `internship-warehouse` (Warehouse daily panel)
- **Time Window:** 2026-03-01 to 2026-03-31 (Mid-panel month: March 2026)

In [ ]:
con.execute("""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM 'hf://datasets/FlyRank/internship-warehouse/*/*2026-03*/*.parquet';
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


- **Feature (5 features):**
  1. `gsc_impressions`: Knowable at decision moment because past impressions are logged up to the report date.
  2. `gsc_avg_position`: Knowable at decision moment because historical search rank is finalized.
  3. `ga4_sessions`: Knowable at decision moment because prior session traffic is recorded.
  4. `ga4_total_engagement_sec`: Knowable at decision moment because prior user engagement time is already tracked.
  5. `sessions_organic`: Knowable at decision moment because organic traffic up to yesterday is locked.

- **Label:** `gsc_clicks` (Predicting search click volume).
- **Context:** `report_date`, `client_hash_id`, `content_hash_id`.
- **Excluded:** Rows where `gsc_data_available IS FALSE` (deliberately dropped because missing GSC integration cannot be modeled).

In [ ]:
con.execute("""
SELECT
    report_date, client_hash_id, content_hash_id,
    gsc_impressions, gsc_clicks, ga4_sessions
FROM 'hf://datasets/FlyRank/internship-warehouse/*/*2026-03*/*.parquet'
LIMIT 5;
""").df()

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,ga4_sessions
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,<NA>


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:

grain_check = con.execute("""
SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS row_count
FROM 'hf://datasets/FlyRank/internship-warehouse/*/*2026-03*/*.parquet'
GROUP BY client_hash_id, content_hash_id, report_date
HAVING COUNT(*) > 1;
""").df()
print(f"1. Grain violations (must be 0): {len(grain_check)}")

# Fact 2: Row count & Date span
span_check = con.execute("""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM 'hf://datasets/FlyRank/internship-warehouse/*/*2026-03*/*.parquet';
""").df()
print("\n2. Row count & Date span:")
display(span_check)

# Fact 3: Availability Check (IS TRUE)
avail_check = con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(CASE WHEN gsc_data_available IS TRUE THEN 1 END) AS available_rows
FROM 'hf://datasets/FlyRank/internship-warehouse/*/*2026-03*/*.parquet';
""").df()
print("\n3. Availability (IS TRUE) check:")
display(avail_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

1. Grain violations (must be 0): 0

2. Row count & Date span:


,total_rows,min_date,max_date
0,9841378,2026-03-01,2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


3. Availability (IS TRUE) check:


,total_rows,available_rows
0,9841378,3611061


In [ ]:
from sklearn.linear_model import Ridge

df_trap = con.execute("""
SELECT
    gsc_impressions,
    gsc_avg_position,
    ga4_sessions,
    ga4_total_engagement_sec,
    sessions_organic,
    gsc_clicks AS target,
    gsc_clicks * 1.0 AS leaked_target_column
FROM 'hf://datasets/FlyRank/internship-warehouse/*/*2026-03*/*.parquet'
WHERE gsc_data_available IS TRUE
LIMIT 10000;
""").df().fillna(0)

y = df_trap['target']

X_leak = df_trap[['gsc_impressions', 'leaked_target_column']]
leak_model = Ridge().fit(X_leak, y)
print("Trap Score (With Leakage):", round(leak_model.score(X_leak, y), 4))

X_clean = df_trap[['gsc_impressions', 'gsc_avg_position', 'ga4_sessions', 'ga4_total_engagement_sec', 'sessions_organic']]
clean_model = Ridge().fit(X_clean, y)
print("Honest Score (Without Leakage):", round(clean_model.score(X_clean, y), 4))

Trap Score (With Leakage): 1.0
Honest Score (Without Leakage): 0.569


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Limitation:**
This slice excludes clients who have not connected Google Search Console (`gsc_data_available IS FALSE`). Consequently, predictions cannot generalize to unverified or newly onboarded client properties lacking active search performance tracking.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.